# Sentinel-1 / Sentinel-2 Acquisition — Tiles 36JTM, 36JUM, 36JTN, 36JUN

**Project:** Deep Learning for Flood Inundation Mapping Using Multi-Source Satellite Data
**Study Area:** KwaZulu-Natal, South Africa
**Flood Event:** April 2022 KwaZulu-Natal Floods
**Author:** Valencia
**Supervisor:** Prof. Innocent Davidson
**Institution:** Cape Peninsula University of Technology (CPUT)



---
## Step 1: Authenticate Earth Engine

In [ ]:
import ee

PROJECT_ID = 'kzn-flood-research'

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

print('Earth Engine authenticated.')
print('Project:', PROJECT_ID)

Earth Engine authenticated.
Project: kzn-flood-research


---
## Step 2: Define Tiles and Candidate Dates

Candidate dates are the ones already used for 36JTM. These are checked
first before considering any alternative dates for 36JUM.

In [ ]:


# flood-polygon coverage: 36JTN and 36JUN.
TILES = ['36JTM', '36JUM', '36JTN', '36JUN']
PRE_FLOOD_DATE  = '2022-03-29'
POST_FLOOD_DATE = '2022-04-28'
CLOUD_FILTER_PCT = 20  # matches the threshold already used for 36JTM

print('Tiles to check       :', TILES)
print('Pre-flood candidate  :', PRE_FLOOD_DATE)
print('Post-flood candidate :', POST_FLOOD_DATE)
print('Cloud filter         : <=', CLOUD_FILTER_PCT, '%')

Tiles to check       : ['36JTM', '36JUM', '36JTN', '36JUN']
Pre-flood candidate  : 2022-03-29
Post-flood candidate : 2022-04-28
Cloud filter         : <= 20 %


---
## Step 3: Check Actual Cloud Cover for Both Tiles on the Candidate Dates

This queries Sentinel-2 L2A metadata directly from Earth Engine — not
assumed, not estimated. `CLOUDY_PIXEL_PERCENTAGE` is the scene-level cloud
cover percentage reported by ESA for each product.

In [ ]:
def check_cloud_cover(tile, date_str, window_days=0):
    """
    Query Sentinel-2 L2A metadata for a given MGRS tile on (or near) a date.
    window_days=0 checks only the exact date; >0 expands the search window.
    Returns a list of (date, cloud_pct, product_id) tuples, sorted by cloud cover.
    """
    from datetime import datetime, timedelta
    center = datetime.strptime(date_str, '%Y-%m-%d')
    start = (center - timedelta(days=window_days)).strftime('%Y-%m-%d')
    end   = (center + timedelta(days=window_days + 1)).strftime('%Y-%m-%d')

    collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterDate(start, end)
                  .filter(ee.Filter.eq('MGRS_TILE', tile)))

    n = collection.size().getInfo()
    if n == 0:
        return []

    # Build a FeatureCollection of metadata (not an ee.List.map, which returns
    # a plain list on getInfo() rather than a {'features': [...]} structure)
    def to_feature(img):
        img = ee.Image(img)
        return ee.Feature(None, {
            'date': img.date().format('YYYY-MM-dd'),
            'cloud_pct': img.get('CLOUDY_PIXEL_PERCENTAGE'),
            'product_id': img.get('PRODUCT_ID')
        })

    fc = ee.FeatureCollection(collection.toList(n).map(to_feature))
    info = fc.getInfo()

    results = [(f['properties']['date'], f['properties']['cloud_pct'], f['properties']['product_id'])
               for f in info['features']]
    return sorted(results, key=lambda x: x[1])

print('Function defined.')

Function defined.


In [ ]:
print('=' * 75)
print('CLOUD COVER CHECK — CANDIDATE DATES (exact match only)')
print('=' * 75)

results_summary = {}
for tile in TILES:
    for label, date_str in [('Pre-flood', PRE_FLOOD_DATE), ('Post-flood', POST_FLOOD_DATE)]:
        scenes = check_cloud_cover(tile, date_str, window_days=0)
        results_summary[(tile, label)] = scenes
        print(f'\nTile {tile} — {label} ({date_str}):')
        if not scenes:
            print('  No scene found on this exact date.')
        for date, cloud_pct, pid in scenes:
            flag = 'OK' if cloud_pct <= CLOUD_FILTER_PCT else 'EXCEEDS THRESHOLD'
            print(f'  {date}  cloud={cloud_pct:.2f}%  [{flag}]  {pid}')

CLOUD COVER CHECK — CANDIDATE DATES (exact match only)

Tile 36JTM — Pre-flood (2022-03-29):
  2022-03-29  cloud=0.03%  [OK]  S2B_MSIL2A_20220329T073609_N0400_R092_T36JTM_20220329T104004

Tile 36JTM — Post-flood (2022-04-28):
  2022-04-28  cloud=0.00%  [OK]  S2B_MSIL2A_20220428T073609_N0400_R092_T36JTM_20220428T105528

Tile 36JUM — Pre-flood (2022-03-29):
  2022-03-29  cloud=0.02%  [OK]  S2B_MSIL2A_20220329T073609_N0400_R092_T36JUM_20220329T104004

Tile 36JUM — Post-flood (2022-04-28):
  2022-04-28  cloud=3.43%  [OK]  S2B_MSIL2A_20220428T073609_N0400_R092_T36JUM_20220428T105528

Tile 36JTN — Pre-flood (2022-03-29):
  2022-03-29  cloud=0.00%  [OK]  S2B_MSIL2A_20220329T073609_N0510_R092_T36JTN_20240523T094629

Tile 36JTN — Post-flood (2022-04-28):
  2022-04-28  cloud=0.00%  [OK]  S2B_MSIL2A_20220428T073609_N0400_R092_T36JTN_20220428T105528

Tile 36JUN — Pre-flood (2022-03-29):
  2022-03-29  cloud=0.03%  [OK]  S2B_MSIL2A_20220329T073609_N0400_R092_T36JUN_20220329T104004

Tile 36JUN — Post

---
## Step 4: Fallback Search for 36JUM (Only if Needed)

If 36JUM does not have a clear scene on the exact candidate date, this
scans a window of ±10 days around it and selects the lowest-cloud
alternative. This only runs for tiles/dates that failed the Step 3 check —
36JTM is not re-queried since its imagery is already acquired and was
filtered at 20% during download.

In [ ]:
FALLBACK_WINDOW_DAYS = 10
final_selection = {}

# 36JTM is already acquired locally — record its dates as fixed, no re-query needed
final_selection[('36JTM', 'Pre-flood')]  = (PRE_FLOOD_DATE, None, 'already acquired (local .SAFE)')
final_selection[('36JTM', 'Post-flood')] = (POST_FLOOD_DATE, None, 'already acquired (local .SAFE)')

# All other tiles are being downloaded manually via Copernicus Browser
# (same workflow as 36JTM), using the dates verified in B1's footprint
# coverage check. This loop confirms each one is genuinely clear before
# you download it.
for tile in ['36JUM', '36JTN', '36JUN']:
    for label, date_str in [('Pre-flood', PRE_FLOOD_DATE), ('Post-flood', POST_FLOOD_DATE)]:
        exact_scenes = results_summary.get((tile, label), [])
        clear_exact = [s for s in exact_scenes if s[1] <= CLOUD_FILTER_PCT]

        if clear_exact:
            chosen = clear_exact[0]
            print(f'{tile} {label}: exact date {date_str} is clear ({chosen[1]:.2f}% cloud). Using it.')
            final_selection[(tile, label)] = chosen
        else:
            print(f'{tile} {label}: exact date {date_str} not clear enough. Searching +/-{FALLBACK_WINDOW_DAYS} days...')
            wide_scenes = check_cloud_cover(tile, date_str, window_days=FALLBACK_WINDOW_DAYS)
            clear_wide = [s for s in wide_scenes if s[1] <= CLOUD_FILTER_PCT]
            if clear_wide:
                chosen = clear_wide[0]
                print(f'  Best alternative: {chosen[0]}  cloud={chosen[1]:.2f}%')
                final_selection[(tile, label)] = chosen
            else:
                print(f'  WARNING: no scene under {CLOUD_FILTER_PCT}% cloud found in the window.')
                final_selection[(tile, label)] = wide_scenes[0] if wide_scenes else None

36JUM Pre-flood: exact date 2022-03-29 is clear (0.02% cloud). Using it.
36JUM Post-flood: exact date 2022-04-28 is clear (3.43% cloud). Using it.
36JTN Pre-flood: exact date 2022-03-29 is clear (0.00% cloud). Using it.
36JTN Post-flood: exact date 2022-04-28 is clear (0.00% cloud). Using it.
36JUN Pre-flood: exact date 2022-03-29 is clear (0.03% cloud). Using it.
36JUN Post-flood: exact date 2022-04-28 is clear (0.08% cloud). Using it.


---
## Step 5: Final Acquisition Summary

In [ ]:
print('=' * 75)
print('FINAL ACQUISITION DATES')
print('=' * 75)
for tile in TILES:
    for label in ['Pre-flood', 'Post-flood']:
        date, cloud, pid = final_selection[(tile, label)]
        cloud_str = f'{cloud:.2f}%' if cloud is not None else 'n/a (pre-existing or manual download)'
        print(f'  {tile}  {label:<11}: {date}   cloud={cloud_str}')
        print(f'           source: {pid}')

FINAL ACQUISITION DATES
  36JTM  Pre-flood  : 2022-03-29   cloud=n/a (pre-existing or manual download)
           source: already acquired (local .SAFE)
  36JTM  Post-flood : 2022-04-28   cloud=n/a (pre-existing or manual download)
           source: already acquired (local .SAFE)
  36JUM  Pre-flood  : 2022-03-29   cloud=0.02%
           source: S2B_MSIL2A_20220329T073609_N0400_R092_T36JUM_20220329T104004
  36JUM  Post-flood : 2022-04-28   cloud=3.43%
           source: S2B_MSIL2A_20220428T073609_N0400_R092_T36JUM_20220428T105528
  36JTN  Pre-flood  : 2022-03-29   cloud=0.00%
           source: S2B_MSIL2A_20220329T073609_N0510_R092_T36JTN_20240523T094629
  36JTN  Post-flood : 2022-04-28   cloud=0.00%
           source: S2B_MSIL2A_20220428T073609_N0400_R092_T36JTN_20220428T105528
  36JUN  Pre-flood  : 2022-03-29   cloud=0.03%
           source: S2B_MSIL2A_20220329T073609_N0400_R092_T36JUN_20220329T104004
  36JUN  Post-flood : 2022-04-28   cloud=0.08%
           source: S2B_MSIL2A_202204

---
## Step 6: Sentinel-2 Acquisition — Manual Download

Sentinel-2 for 36JUM, 36JTN, and 36JUN was downloaded manually via the
Copernicus Browser (same workflow as the existing 36JTM imagery), using
the dates and product IDs confirmed above. This keeps all four tiles in
the same raw product format (`.SAFE`) and avoids relying on an
Earth-Engine-side export step for this stage.

Product IDs used:

| Tile | Pre-flood (2022-03-29) | Post-flood (2022-04-28) |
|---|---|---|
| 36JUM | S2B_MSIL2A_20220329T073609_N0400_R092_T36JUM_20220329T104004 | S2B_MSIL2A_20220428T073609_N0400_R092_T36JUM_20220428T105528 |
| 36JTN | S2B_MSIL2A_20220329T073609_N0510_R092_T36JTN_20240523T094629 | S2B_MSIL2A_20220428T073609_N0400_R092_T36JTN_20220428T105528 |
| 36JUN | S2B_MSIL2A_20220329T073609_N0400_R092_T36JUN_20220329T104004 | S2B_MSIL2A_20220428T073609_N0400_R092_T36JUN_20220428T105528 |

Downloaded files are placed in the same folder as the existing 36JTM raw
data (`KZN_Research_Colab/Sentinel2/Raw/`), preserving the original
Copernicus `.SAFE` naming convention so all four tiles are easy to glob
together in the mosaic step.

---
## Step 7: Acquire Sentinel-1 (VV, VH) for All Four Tiles, Both Dates
Sentinel-1 imagery is acquired directly from Earth Engine for all four
required tiles (36JTM, 36JUM, 36JTN, 36JUN), matching the same
pre-flood and post-flood acquisition dates used for Sentinel-2. Each
export uses the actual acquired Sentinel-2 footprint for that tile as
the export region, ensuring SAR and optical coverage align precisely.

In [ ]:
S1_OUT_DIR_PARENT = 'KZN_Research_Colab/Sentinel1_v2'

def get_s1_image(tile_geom, date_str, window_days=6):
    """
    Sentinel-1 revisit is ~6 or 12 days, so an exact-date match is unlikely.
    A window around the target date is used to find the nearest available pass.
    """
    from datetime import datetime, timedelta
    center = datetime.strptime(date_str, '%Y-%m-%d')
    start = (center - timedelta(days=window_days)).strftime('%Y-%m-%d')
    end   = (center + timedelta(days=window_days)).strftime('%Y-%m-%d')

    collection = (ee.ImageCollection('COPERNICUS/S1_GRD')
                  .filterBounds(tile_geom)
                  .filterDate(start, end)
                  .filter(ee.Filter.eq('instrumentMode', 'IW'))
                  .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                  .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')))

    n = collection.size().getInfo()
    if n == 0:
        return None, None
    img = collection.sort('system:time_start').first()
    actual_date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    return img.select(['VV', 'VH']), actual_date

# Use each tile's ACTUAL acquired Sentinel-2 footprint (from B1's verification
# step) as the export region, so Sentinel-1 covers exactly the same ground
# as the optical imagery for that tile — not an approximate bounding box.
def get_s2_footprint(tile, date_str):
    img = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterDate(date_str, ee.Date(date_str).advance(1, 'day'))
           .filter(ee.Filter.eq('MGRS_TILE', tile))
           .first())
    return img.geometry()

tile_geoms = {tile: get_s2_footprint(tile, PRE_FLOOD_DATE) for tile in TILES}

s1_export_tasks = []
for tile in TILES:
    for label, target_date in [('PreFlood', PRE_FLOOD_DATE), ('PostFlood', POST_FLOOD_DATE)]:
        img, actual_date = get_s1_image(tile_geoms[tile], target_date)
        if img is None:
            print(f'{tile} {label}: no Sentinel-1 scene found near {target_date}')
            continue

        task_name = f'S1_{tile}_{label}_{actual_date.replace("-", "")}'
        task = ee.batch.Export.image.toDrive(
            image=img,
            description=task_name,
            folder=f'{S1_OUT_DIR_PARENT}/{tile}',
            fileNamePrefix=task_name,
            scale=10,
            region=tile_geoms[tile],
            maxPixels=1e10
        )
        task.start()
        s1_export_tasks.append((task_name, task))
        print(f'Started export: {task_name}  (target {target_date}, actual pass {actual_date})')

Started export: S1_36JTM_PreFlood_20220324  (target 2022-03-29, actual pass 2022-03-24)
Started export: S1_36JTM_PostFlood_20220422  (target 2022-04-28, actual pass 2022-04-22)
Started export: S1_36JUM_PreFlood_20220324  (target 2022-03-29, actual pass 2022-03-24)
Started export: S1_36JUM_PostFlood_20220424  (target 2022-04-28, actual pass 2022-04-24)
Started export: S1_36JTN_PreFlood_20220324  (target 2022-03-29, actual pass 2022-03-24)
Started export: S1_36JTN_PostFlood_20220422  (target 2022-04-28, actual pass 2022-04-22)
Started export: S1_36JUN_PreFlood_20220324  (target 2022-03-29, actual pass 2022-03-24)
Started export: S1_36JUN_PostFlood_20220424  (target 2022-04-28, actual pass 2022-04-24)


**Step 8: DEM Acquisition**


In [ ]:

from shapely.geometry import shape, mapping
from shapely.ops import unary_union

def get_s2_footprint_shapely(tile, date_str):
    img = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterDate(date_str, ee.Date(date_str).advance(1, 'day'))
           .filter(ee.Filter.eq('MGRS_TILE', tile))
           .first())
    return shape(img.geometry().getInfo())

footprints_4 = {t: get_s2_footprint_shapely(t, PRE_FLOOD_DATE) for t in TILES}
combined_4 = unary_union(list(footprints_4.values()))

print('Rebuilt 4-tile footprint union for:', TILES)
print('Combined bounds:', combined_4.bounds)

Rebuilt 4-tile footprint union for: ['36JTM', '36JUM', '36JTN', '36JUN']
Combined bounds: (29.86502916143537, -30.81587278479146, 32.07451164142597, -28.892503651216654)


In [ ]:
dem_source = ee.Image('USGS/SRTMGL1_003')  # native 30m SRTM
dem_export_region = ee.Geometry(mapping(combined_4))

dem_task = ee.batch.Export.image.toDrive(
    image=dem_source,
    description='DEM_4tile_30m_native',
    folder='KZN_Research_Colab/DEM/Processed',
    fileNamePrefix='DEM_4tile_30m_native',
    scale=30,
    region=dem_export_region,
    maxPixels=1e10
)
dem_task.start()
print('Started DEM export: DEM_4tile_30m_native (native 30m, no resampling)')

Started DEM export: DEM_4tile_30m_native (native 30m, no resampling)


In [ ]:
import time

print('Monitoring DEM export task...')
while True:
    status = dem_task.status()['state']
    print(f'  Status: {status}')
    if status in ('COMPLETED', 'FAILED', 'CANCELLED'):
        break
    time.sleep(30)

print()
if status == 'COMPLETED':
    print('DEM export finished successfully.')
elif status == 'FAILED':
    print('DEM export FAILED. Error details:')
    print(dem_task.status())
else:
    print(f'DEM export ended with status: {status}')

Monitoring DEM export task...
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: COMPLETED

DEM export finished successfully.


---
## Step 9: Acquisition Summary

This becomes the documented data acquisition record for the methodology
chapter — every date and source is traceable to this notebook's output.

In [ ]:
print('=' * 75)
print('DATA ACQUISITION SUMMARY')
print('=' * 75)
print()
print('Sentinel-2:')
print(f'  36JTM Pre-flood  : {PRE_FLOOD_DATE}  (pre-existing download, 20% cloud filter)')
print(f'  36JTM Post-flood : {POST_FLOOD_DATE}  (pre-existing download, 20% cloud filter)')
jum_pre  = final_selection[('36JUM', 'Pre-flood')]
jum_post = final_selection[('36JUM', 'Post-flood')]
print(f'  36JUM Pre-flood  : {jum_pre[0]}  cloud={jum_pre[1]}')
print(f'  36JUM Post-flood : {jum_post[0]}  cloud={jum_post[1]}')
print()
print('Sentinel-1:')
for name, t in s1_export_tasks:
    print(f'  {name}: {t.status()["state"]}')
print()
print('Next step: mosaic 36JTM + 36JUM into a single aligned grid')
print('(B3_mosaic_tiles.ipynb)')

DATA ACQUISITION SUMMARY

Sentinel-2:
  36JTM Pre-flood  : 2022-03-29  (pre-existing download, 20% cloud filter)
  36JTM Post-flood : 2022-04-28  (pre-existing download, 20% cloud filter)
  36JUM Pre-flood  : 2022-03-29  cloud=0.015698
  36JUM Post-flood : 2022-04-28  cloud=3.425309

Sentinel-1:
  S1_36JTM_PreFlood_20220324: COMPLETED
  S1_36JTM_PostFlood_20220422: COMPLETED
  S1_36JUM_PreFlood_20220324: COMPLETED
  S1_36JUM_PostFlood_20220424: COMPLETED
  S1_36JTN_PreFlood_20220324: COMPLETED
  S1_36JTN_PostFlood_20220422: COMPLETED
  S1_36JUN_PreFlood_20220324: COMPLETED
  S1_36JUN_PostFlood_20220424: COMPLETED

Next step: mosaic 36JTM + 36JUM into a single aligned grid
(B3_mosaic_tiles.ipynb)


---
## Step 7 (Revised): Re-acquiring Sentinel-1 Post-Flood

The original post-flood export (target: 28 April) matched the nearest
available pass, which turned out to only partially cover the study area
(51-90% nodata per tile). A direct check of swath footprints found
**17 April 2022 (orbit 43, ascending)** gives 98.8% true coverage — this
date is re-exported here instead. Note: 17 April is within the flood
event window, but differs from the Sentinel-2 post-flood date (28 April).

In [ ]:
# Check if there were OTHER S1 passes near the post-flood target date that might have fuller coverage
post_candidates = (ee.ImageCollection('COPERNICUS/S1_GRD')
                    .filterBounds(tile_geoms['36JTM'])
                    .filterDate('2022-04-15', '2022-05-05')
                    .filter(ee.Filter.eq('instrumentMode', 'IW'))
                    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')))

n = post_candidates.size().getInfo()
info = post_candidates.toList(n).map(lambda img: ee.Feature(None, {
    'date': ee.Image(img).date().format('YYYY-MM-dd'),
    'orbit': ee.Image(img).get('relativeOrbitNumber_start'),
    'pass': ee.Image(img).get('orbitProperties_pass'),
}))
results = ee.FeatureCollection(info).getInfo()['features']

print(f'{n} Sentinel-1 passes found near 36JTM, 15 April - 5 May 2022:\n')
for f in results:
    p = f['properties']
    print(f"  {p['date']}  orbit={p['orbit']}  pass={p['pass']}")

10 Sentinel-1 passes found near 36JTM, 15 April - 5 May 2022:

  2022-04-17  orbit=43  pass=ASCENDING
  2022-04-17  orbit=43  pass=ASCENDING
  2022-04-20  orbit=79  pass=DESCENDING
  2022-04-22  orbit=116  pass=ASCENDING
  2022-04-22  orbit=116  pass=ASCENDING
  2022-04-27  orbit=6  pass=DESCENDING
  2022-04-29  orbit=43  pass=ASCENDING
  2022-04-29  orbit=43  pass=ASCENDING
  2022-05-04  orbit=116  pass=ASCENDING
  2022-05-04  orbit=116  pass=ASCENDING


In [ ]:
TARGET_AREA = ee.Geometry.Rectangle([30.77, -30.82, 32.07, -29.71])  # full 4-tile bounding area

post_candidates = (ee.ImageCollection('COPERNICUS/S1_GRD')
                    .filterBounds(TARGET_AREA)
                    .filterDate('2022-04-15', '2022-05-05')
                    .filter(ee.Filter.eq('instrumentMode', 'IW'))
                    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')))

n = post_candidates.size().getInfo()
image_list = post_candidates.toList(n)

print(f'Checking actual footprint coverage for {n} candidate S1 passes:\n')
for i in range(n):
    img = ee.Image(image_list.get(i))
    date = img.date().format('YYYY-MM-dd').getInfo()
    orbit = img.get('relativeOrbitNumber_start').getInfo()
    direction = img.get('orbitProperties_pass').getInfo()

    footprint = img.geometry()
    intersection_area = footprint.intersection(TARGET_AREA, 1).area().getInfo()
    target_area = TARGET_AREA.area().getInfo()
    coverage_pct = 100 * intersection_area / target_area

    print(f'  {date}  orbit={orbit}  {direction}  coverage={coverage_pct:.1f}% of 4-tile area')

Checking actual footprint coverage for 6 candidate S1 passes:

  2022-04-17  orbit=43  ASCENDING  coverage=98.8% of 4-tile area
  2022-04-20  orbit=79  DESCENDING  coverage=61.1% of 4-tile area
  2022-04-24  orbit=145  ASCENDING  coverage=44.8% of 4-tile area
  2022-04-27  orbit=6  DESCENDING  coverage=20.9% of 4-tile area
  2022-04-27  orbit=6  DESCENDING  coverage=68.2% of 4-tile area
  2022-04-29  orbit=43  ASCENDING  coverage=98.8% of 4-tile area


In [ ]:
POST_FLOOD_S1_DATE = '2022-04-17'  # orbit 43, ASCENDING, 98.8% coverage of full 4-tile area

s1_export_tasks_v2 = []
for tile in TILES:
    img, actual_date = get_s1_image(tile_geoms[tile], POST_FLOOD_S1_DATE, window_days=1)
    if img is None:
        print(f'{tile}: no scene found on {POST_FLOOD_S1_DATE}')
        continue

    task_name = f'S1_{tile}_PostFlood_v2_{actual_date.replace("-", "")}'
    task = ee.batch.Export.image.toDrive(
        image=img,
        description=task_name,
        folder=f'KZN_Research_Colab/Sentinel1_v2/{tile}',
        fileNamePrefix=task_name,
        scale=10,
        region=tile_geoms[tile],
        maxPixels=1e10
    )
    task.start()
    s1_export_tasks_v2.append((task_name, task))
    print(f'Started export: {task_name}  (target {POST_FLOOD_S1_DATE}, actual pass {actual_date})')

Started export: S1_36JTM_PostFlood_v2_20220417  (target 2022-04-17, actual pass 2022-04-17)
Started export: S1_36JUM_PostFlood_v2_20220417  (target 2022-04-17, actual pass 2022-04-17)
Started export: S1_36JTN_PostFlood_v2_20220417  (target 2022-04-17, actual pass 2022-04-17)
Started export: S1_36JUN_PostFlood_v2_20220417  (target 2022-04-17, actual pass 2022-04-17)


In [ ]:
import time
while True:
    statuses = [(name, t.status()['state']) for name, t in s1_export_tasks_v2]
    print(statuses)
    if all(s in ('COMPLETED', 'FAILED', 'CANCELLED') for _, s in statuses):
        break
    time.sleep(30)
print('\nAll done.')
for name, t in s1_export_tasks_v2:
    print(f'  {name}: {t.status()["state"]}')

[('S1_36JTM_PostFlood_v2_20220417', 'READY'), ('S1_36JUM_PostFlood_v2_20220417', 'READY'), ('S1_36JTN_PostFlood_v2_20220417', 'READY'), ('S1_36JUN_PostFlood_v2_20220417', 'READY')]
[('S1_36JTM_PostFlood_v2_20220417', 'READY'), ('S1_36JUM_PostFlood_v2_20220417', 'READY'), ('S1_36JTN_PostFlood_v2_20220417', 'READY'), ('S1_36JUN_PostFlood_v2_20220417', 'READY')]
[('S1_36JTM_PostFlood_v2_20220417', 'READY'), ('S1_36JUM_PostFlood_v2_20220417', 'READY'), ('S1_36JTN_PostFlood_v2_20220417', 'READY'), ('S1_36JUN_PostFlood_v2_20220417', 'READY')]
[('S1_36JTM_PostFlood_v2_20220417', 'READY'), ('S1_36JUM_PostFlood_v2_20220417', 'READY'), ('S1_36JTN_PostFlood_v2_20220417', 'READY'), ('S1_36JUN_PostFlood_v2_20220417', 'READY')]
[('S1_36JTM_PostFlood_v2_20220417', 'READY'), ('S1_36JUM_PostFlood_v2_20220417', 'READY'), ('S1_36JTN_PostFlood_v2_20220417', 'READY'), ('S1_36JUN_PostFlood_v2_20220417', 'READY')]
[('S1_36JTM_PostFlood_v2_20220417', 'READY'), ('S1_36JUM_PostFlood_v2_20220417', 'READY'), ('S1

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Re-export Sentinel-1 post-flood for 36JUN only, using the best available
# pass: 27 April 2022, orbit 6, DESCENDING (77.9% coverage of the tile,
# vs. the original 24 April export which left 64% as nodata).
# Note: JUN contains no UNOSAT flood polygons, so full coverage was not
# required here - this improves data completeness without blocking
# the pipeline on a tile that doesn't affect label accuracy.

JUN_FIX_DATE = '2022-04-27'

img, actual_date = get_s1_image(tile_geoms['36JUN'], JUN_FIX_DATE, window_days=1)

if img is None:
    print('No scene found - check date/window')
else:
    task_name = f'S1_36JUN_PostFlood_v3_{actual_date.replace("-", "")}'
    task = ee.batch.Export.image.toDrive(
        image=img,
        description=task_name,
        folder='KZN_Research_Colab/Sentinel1_v2/36JUN',
        fileNamePrefix=task_name,
        scale=10,
        region=tile_geoms['36JUN'],
        maxPixels=1e10
    )
    task.start()
    print(f'Started export: {task_name}  (target {JUN_FIX_DATE}, actual pass {actual_date})')

Started export: S1_36JUN_PostFlood_v3_20220427  (target 2022-04-27, actual pass 2022-04-27)


In [ ]:
import time
while True:
    status = task.status()['state']
    print(status)
    if status in ('COMPLETED', 'FAILED', 'CANCELLED'):
        break
    time.sleep(30)
print('\nDone:', task.status())

RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
COMPLETED

Done: {'state': 'COMPLETED', 'description': 'S1_36JUN_PostFlood_v3_20220427', 'priority': 100, 'creation_timestamp_ms': 1781922854738, 'update_timestamp_ms': 1781923534858, 'start_timestamp_ms': 1781922860490, 'task_type': 'EXPORT_IMAGE', 'destination_uris': ['https://drive.google.com/#folders/1vz9t6LSzbfNqBapt1UuRB5TYlplxV1oR'], 'attempt': 1, 'batch_eecu_usage_seconds': 296.8999938964844, 'id': 'ZFRSCTMTSO5K6E5XD5JZZKHI', 'name': 'projects/kzn-flood-research/operations/ZFRSCTMTSO5K6E5XD5JZZKHI'}


In [ ]:
import rasterio
import numpy as np
from rasterio.windows import Window

jun_v3_path = ROOT + 'Sentinel1_v2/36JUN/S1_36JUN_PostFlood_v3_20220427.tif'

with rasterio.open(jun_v3_path) as src:
    nan_count = 0
    total_count = 0
    for row_start in range(0, src.height, 1000):
        rows = min(1000, src.height - row_start)
        window = Window(0, row_start, src.width, rows)
        chunk = src.read(1, window=window)
        nan_count += np.isnan(chunk).sum()
        total_count += chunk.size
    pct = 100 * nan_count / total_count
    print(f'36JUN v3: {pct:.2f}% NaN  shape={src.shape}')

36JUN v3: 22.25% NaN  shape=(10981, 10981)
